# Signal Research Notebook

Interactive research and development notebook for signal creation and testing.

## Workflow
1. **Load & Explore Data** - Load market data and inspect distributions
2. **Develop Signal** - Build and test signal logic interactively
3. **Implement** - Copy validated logic to `create_signal.py`
4. **Execute** - Run `uv run create-signal` to generate `data/signal.parquet`
5. **Validate Signal** - use `uv run ew-dash` to view signal characteristics.
6. **Backtest** - Use `uv run backtest` for slurm backtest on super computer
7. **Performance** - Use `uv run opt-dash` for in depth analysis of mvo backtested signal.

## Tips
- Use cells to isolate different aspects of your signal
- Modify parameters directly in cells to test variations
- Check signal statistics regularly to catch issues early
- Document your assumptions and findings as you develop

## Setup

In [ ]:
import polars as pl
import numpy as np
import datetime as dt
import sf_quant.data as sfd
import sf_quant.research as sfr
import polars_ols

## 1. Load & Explore Data

Load your market data and inspect key characteristics before developing the signal.

In [ ]:
#to be able to run this signal you will need to get the crsp and compustat merged data file
#from wrds. 
data = pl.read_csv('crsp_comp_merged.csv', dtypes={"CUSIP": pl.Utf8})

start_date = "1970-01-01"
end_date   = "2004-12-31"

data = data.filter(
    (pl.col("MthCalDt") >= start_date) & 
    (pl.col("MthCalDt") <= end_date)
)

data

## 2. Signal Development

Build and test your signal logic. Modify parameters and logic here to find optimal configurations.

In [ ]:
# parse Compustat date
df = data.with_columns(
    pl.col("datadate").str.to_date("%Y-%m-%d")
)

# keep only active industrial firms (as done in the paper)
df = df.filter(
    (pl.col("costat") == "A") &
    (pl.col("indfmt") == "INDL") &
    (pl.col("datafmt") == "STD") &
    (pl.col("popsrc") == "D") &
    (pl.col("consol") == "C")
)

# collapse to one row per firm-year
df_fy = (
    df.sort(["gvkey", "datadate"])
      .unique(subset=["gvkey", "fyear"], keep="last")
      .sort(["gvkey", "fyear"])
)

# explicit firm-year lags
df_fy = df_fy.with_columns([
    pl.col("act").shift(1).over("gvkey").alias("act_lag"),
    pl.col("che").shift(1).over("gvkey").alias("che_lag"),
    pl.col("rect").shift(1).over("gvkey").alias("rect_lag"),
    pl.col("invt").shift(1).over("gvkey").alias("invt_lag"),
    pl.col("lct").shift(1).over("gvkey").alias("lct_lag"),
    pl.col("ap").shift(1).over("gvkey").alias("ap_lag"),
    pl.col("dlc").shift(1).over("gvkey").alias("dlc_lag"),
    pl.col("txp").shift(1).over("gvkey").alias("txp_lag"),
    pl.col("at").shift(1).over("gvkey").alias("at_lag"),
])

# changes in current assets and liabilities
df_fy = df_fy.with_columns([
    (pl.col("act") - pl.col("act_lag")).alias("d_act"),
    (pl.col("che") - pl.col("che_lag")).alias("d_che"),
    (pl.col("lct") - pl.col("lct_lag")).alias("d_lct"),
    (pl.col("dlc") - pl.col("dlc_lag")).alias("d_dlc"),
    (pl.col("txp") - pl.col("txp_lag")).alias("d_txp"),
])

# average total assets
df_fy = df_fy.with_columns(
    ((pl.col("at") + pl.col("at_lag")) / 2).alias("avg_at")
)

df_fy = df_fy.with_columns([
    (pl.col("d_act") - pl.col("d_che")).alias("d_ca"),
    (pl.col("d_lct") - pl.col("d_dlc") - pl.col('d_txp')).alias("d_cl")
])

#creation of the accruals signal
df_fy = df_fy.with_columns(
    pl.when(pl.col("avg_at") > 0)
      .then((pl.col("d_ca") - pl.col("d_cl") - pl.col("dp")) / pl.col("avg_at"))
      .otherwise(None)
      .alias("accruals")
)

# # Sloan accruals
# df_fy = df_fy.with_columns(
#     (
#         ((pl.col("d_act") - pl.col("d_che"))
#          - (pl.col("d_lct") - pl.col("d_dlc") - pl.col("d_txp"))
#          - pl.col("dp")
#         ) / pl.col("avg_at")
#     ).alias("accruals")
# )

# scaled earnings and cash flow
df_fy = df_fy.with_columns([
    (pl.col("oiadp") / pl.col("avg_at")).alias("scaled_earnings"),
    ((pl.col("oiadp") / pl.col("avg_at")) - pl.col("accruals")).alias("cash_flow"),
])

# drop first firm-year + kill bad denominators and infinities
df_fy = df_fy.filter(
    (pl.col("avg_at") > 0) & pl.col("accruals").is_finite()
)

# Table I variables
df_fy = df_fy.with_columns([
    ((pl.col("d_act") - pl.col("d_che")) / pl.col("avg_at")).alias("delta_ca"),
    ((pl.col("d_lct") - pl.col("d_dlc") - pl.col("d_txp")) / pl.col("avg_at")).alias("delta_cl"),
    (pl.col("dp") / pl.col("avg_at")).alias("depreciation"),
])

# here I select the cols that i want put back into monthly space
monthly_vars = ["delta_ca", "delta_cl", "depreciation", "accruals", "scaled_earnings", "cash_flow"]

#simply join those cols to the og df
df_monthly = df.join(
    df_fy.select(["gvkey", "fyear"] + monthly_vars),
    on=["gvkey", "fyear"],
    how="left"
)

#should be the same table as maxwells (very close to the paper's table)
cols = monthly_vars
table_I = pl.DataFrame({
    "variable": cols,
    "mean": [df_monthly[c].mean() for c in cols],
    "std": [df_monthly[c].std() for c in cols],
    "p25": [df_monthly[c].quantile(0.25) for c in cols],
    "median": [df_monthly[c].median() for c in cols],
    "p75": [df_monthly[c].quantile(0.75) for c in cols],
})

table_I
# df_monthly.head(20)


In [ ]:
#drops infinites
data = df_monthly.filter(pl.col("accruals").is_finite())

# data = data.with_columns(
#     pl.col('accruals').qcut(5, labels=['0','1','2','3','4'], allow_duplicates=True).over('MthCalDt').alias('acc_bins')
# )

# data = data.filter(pl.col("acc_bins").is_not_null())

# split earnings into 5 bins
# data = data.with_columns(
#     pl.col('scaled_earnings').qcut(5, labels=['0','1','2','3','4'], allow_duplicates=True).over(['MthCalDt', 'acc_bins']).alias('earn_bins')
# )


#****This here is testing the double sort with mktcap to see if accruals still lived on in 
#high or low mktcap stocks specifically*****

data = data.with_columns((pl.col('MthPrc').abs() * pl.col('ShrOut') * 1000).alias('mktcap'))

data = data.with_columns(
    pl.col('mktcap').qcut(5, labels=['0','1','2','3','4'], allow_duplicates=True).over('MthCalDt').alias('mktcap_bins')
)

data = data.filter(pl.col("mktcap_bins").is_not_null())

#testing a double sort based on mktcap and accruals
data = data.with_columns(
    pl.col('accruals').qcut(5, labels=['0','1','2','3','4'], allow_duplicates=True).over(['MthCalDt', 'mktcap_bins']).alias('acc_bins')
)

data = data.filter(pl.col("acc_bins").is_not_null())

data

In [ ]:
#create double sort table
table = (
    data
    .group_by(["acc_bins", "mktcap_bins"])
    .agg(pl.col("MthRet").mean().alias("mean_ret"))
    .sort(["acc_bins", "mktcap_bins"])
)

table_5x5 = table.pivot(
    values="mean_ret",
    index="mktcap_bins",
    columns="acc_bins"
)

data = data.filter(
    pl.col("accruals").is_finite() &
    pl.col("scaled_earnings").is_finite() &
    pl.col("MthRet").is_finite()
)

table_5x5

In [ ]:
data = data.with_columns(
    pl.col("MthCalDt").str.strptime(pl.Date, "%Y-%m-%d").alias("date")
)

#fama french data
start = dt.date(1970, 1, 1)
end   = dt.date(1994, 12, 31)

ff_data = sfd.load_fama_french(start=start, end=end)
# ff_data.head()

data = data.join(ff_data, on='date', how='inner')

# data

#for merging in next box
acc_df = data.select(['MthCalDt', 'acc_bins'])

acc_df

In [ ]:
#the regression of df to get the residuals
pdf = data.to_pandas()

def get_residuals(df):
    y = df['MthRet'] - df['rf']
    X = df[['mkt_rf', 'smb', 'hml', 'rmw', 'cma']]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    df = df.copy()
    df['residual'] = model.resid
    return df

#get resids from df
pdf = pdf.groupby('acc_bins', group_keys=False).apply(get_residuals)

#joining cause groupby destroys acc_bins so im putting back in
data_res = pl.from_pandas(pdf)
data_res = data_res.join(acc_df, how='inner', on='MthCalDt')


#mean residual per bin per month
bin_ret = data_res.group_by(["date", "acc_bins"]).agg(pl.col("residual").mean().alias("bin_ret"))

#turns rows to cols
bin_ret = bin_ret.pivot(values="bin_ret",index="date",columns="acc_bins")

#spread
bin_ret = bin_ret.with_columns((pl.col("0") - pl.col("4")).alias("spread"))

#create new cols for vol scaled residuals
bin_vol_scaled = bin_ret.with_columns([
    (pl.col(str(b)) / pl.col(str(b)).rolling_std(12)).alias(f"{b}_vol_scaled")
    for b in range(5)] + [
    (pl.col("spread") / pl.col("spread").rolling_std(12)).alias("spread_vol_scaled")
])

#to pandas for graphing
pdf = bin_vol_scaled.to_pandas()




In [ ]:
# for b in range(5):
#     plt.plot(
#         pdf['date'],
#         pdf[f'{b}_vol_scaled'].rolling(12).mean(),
#         label=f'{b}'
#     )


# for b in range(5):
#     plt.plot(
#         pdf['date'],
#         pdf[f'{b}_vol_scaled'].cumsum(),
#         label=f'{b}'
#     )

# plt.plot(
#     pdf['date'],
#     pdf['spread_vol_scaled'].cumsum(),
#     label='spread',
#     color='black'
# )

for b in range(5):
    plt.plot(pdf['date'], pdf[f'{b}_vol_scaled'], label=f'{b}')

plt.plot(pdf['date'], pdf['spread_vol_scaled'], label='spread', color='black')
plt.xlabel("date")
plt.ylabel("vol scaled residual")
plt.title("vol scaled residuals per accrual bin")
plt.legend()
plt.show()

## 3. Signal Analysis

Examine signal statistics and distributions to understand its characteristics.

### Statistics

In [ ]:
#tests to see if any good at all

spread_mean = pdf["spread"].mean()
spread_t = spread_mean / (pdf["spread"].std() / np.sqrt(len(pdf)))

# print(spread_t)

sharpe = pdf["spread"].mean() / pdf["spread"].std()
# print(sharpe)

# data_res = data_res.to_pandas()
# data_res.groupby("acc_bins")["residual"].mean()

# print(f'Spread t-stat: {spread_t.round(2)}')
print(f'Sharpe Ratio: {sharpe.round(2)}')


#***** not good at all btw *****